# Self-Derived Skills: Colab runner

Setup, needed once per account:

1. Select a GPU in **Runtime > Change runtime type**.
2. Create a fine-grained GitHub token limited to this repository, with read-only Contents access.
3. Add it in Colab's **Secrets** panel (the key icon) as `GH_TOKEN`, and turn on notebook access.

Then choose **Runtime > Run all**. Any failed step stops the notebook with an error. The probe writes to the runtime's local disk, so its files disappear when the session ends.

In [ ]:
import shutil

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU found. Select one in Runtime > Change runtime type, then run all.")

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
import subprocess

from google.colab import userdata

REPO_URL = "https://github.com/Ruichu-Xia/self-derived-skills.git"
REPO_DIR = "/content/self-derived-skills"
RUN_ROOT = "/content/runs"

# The token stays in an environment variable so that it never appears in a command, URL, or error.
GIT_ENV = {**os.environ, "GH_TOKEN": userdata.get("GH_TOKEN")}
CREDENTIAL_HELPER = (
    "credential.helper=!f() { echo username=x-access-token; echo password=$GH_TOKEN; }; f"
)


def git(*args, cwd=None):
    """Run a git command that authenticates with the token in the environment."""
    subprocess.run(["git", "-c", CREDENTIAL_HELPER, *args], env=GIT_ENV, cwd=cwd, check=True)


if os.path.isdir(REPO_DIR):
    git("pull", "--ff-only", cwd=REPO_DIR)
else:
    git("clone", REPO_URL, REPO_DIR)

os.chdir(REPO_DIR)
!git log -1 --format='%h %s'

In [ ]:
%pip install -q -e ".[train,cuda]"
# Colab preinstalls a torchao too old for PEFT, which then crashes. This project does not use it.
%pip uninstall -q -y torchao

# A fresh process confirms that the packages import, without loading them into this kernel.
subprocess.run(
    [
        "python",
        "-c",
        "import fla, peft, torch, transformers, trl; "
        "from peft.import_utils import is_torchao_available; "
        "assert torch.cuda.is_available() and not is_torchao_available(); "
        "print('torch', torch.__version__, 'transformers', transformers.__version__, "
        "'trl', trl.__version__, 'peft', peft.__version__)",
    ],
    check=True,
)

## GRPO probe

Checks that the model loads, that left padding does not change its outputs, and how long a real GRPO step takes. It runs 6 training steps and trains nothing useful. Send the printed summary back when it finishes.

In [ ]:
from datetime import datetime
from pathlib import Path

PROBE_DIR = Path(RUN_ROOT) / f"probe-{datetime.now():%Y%m%d-%H%M%S}"
!python experiments/probe_grpo.py --config configs/grpo_math.yaml --output-dir "{PROBE_DIR}" --max-steps 6

summary_path = PROBE_DIR / "probe_summary.json"
if not summary_path.exists():
    raise RuntimeError("The probe failed, so there is no summary. See the traceback above.")
print(summary_path.read_text())